In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.6 MB/s eta 0:00:00a 0:00:01


In [3]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import glob
import os
import re
from collections import deque, defaultdict
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [12]:
IMG_DIR = "/kaggle/input/datasets/axelpham/graduation/data/data/images/train"
LBL_DIR = "/kaggle/input/datasets/axelpham/graduation/data/data/labels/train"
YOLO_PATH = "/kaggle/input/models/axelpham/yolo26-v2/pytorch/default/1/best.pt"
SAVE_LSTM_PATH = "/kaggle/working/lstm_intent.pth"

In [5]:
STATE_WINDOW = 10      # Độ dài chuỗi (16 frames)
FRAME_SKIP = 3         # Lấy mẫu thưa (cứ 3 frame lấy 1)
BATCH_SIZE = 64
EPOCHS = 50

In [6]:
class IntentLSTM(nn.Module):
    def __init__(self, input_size=5, hidden_size=64, num_classes=2):
        super(IntentLSTM, self).__init__()
        # Input size = 5 (cx, cy, w, h, conf)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        # x shape: (Batch, Seq_Len, Features) -> (B, 10, 5)
        out, _ = self.lstm(x)
        # Lấy output của time-step cuối cùng
        return self.fc(out[:, -1, :])

# Hàm hỗ trợ sắp xếp tự nhiên
def natural_sort_key(filename):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', os.path.basename(filename))]

In [7]:
yolo = YOLO(YOLO_PATH)
image_files = sorted(glob.glob(os.path.join(IMG_DIR, '*.jpg')), key=natural_sort_key)

full_tracks = defaultdict(list)
for i, img_path in enumerate(image_files):
    frame = cv2.imread(img_path)
    if frame is None: continue
    h_img, w_img = frame.shape[:2]
    
    # Đọc Ground Truth (Có nhãn Cut-in hay không)
    gt_cutin = False
    lbl_path = os.path.join(LBL_DIR, os.path.splitext(os.path.basename(img_path))[0] + ".txt")
    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            for line in f:
                if int(float(line.split()[0])) == 4: # Class 4 = Cut-in
                    gt_cutin = True; break

    results = yolo.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        
        for box, tid, conf in zip(boxes, ids, confs):
            x1, y1, x2, y2 = box
            feature = [
                ((x1 + x2) / 2) / w_img, # cx
                ((y1 + y2) / 2) / h_img, # cy
                (x2 - x1) / w_img,       # w
                (y2 - y1) / h_img,       # h
                float(conf)
            ]
            full_tracks[tid].append({'feature': feature, 'label': gt_cutin, 'idx': i})

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 257ms
Prepared 1 package in 93ms
Installed 1 package in 6ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



In [8]:
X_data, Y_data = [], []
REQ_LEN = (STATE_WINDOW - 1) * FRAME_SKIP + 1

for tid, track in full_tracks.items():
    if len(track) < REQ_LEN: continue
    
    for i in range(len(track) - REQ_LEN + 1):
        window = track[i : i + REQ_LEN]
        sampled = window[::FRAME_SKIP] # Lấy mẫu thưa
        
        features = [item['feature'] for item in sampled]
        # Nếu trong chuỗi tương lai gần có Cut-in thì gán nhãn 1
        label = 1 if any(item['label'] for item in window) else 0 
        
        X_data.append(features)
        Y_data.append(label)

X_tensor = torch.FloatTensor(np.array(X_data)) # (N, 10, 5)
Y_tensor = torch.LongTensor(np.array(Y_data))  # (N,)

dataset = TensorDataset(X_tensor, Y_tensor)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"Đã tạo {len(dataset)} mẫu chuỗi hành vi.")

Đã tạo 28299 mẫu chuỗi hành vi.


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = IntentLSTM().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
for epoch in range(EPOCHS):
    total_loss = 0
    correct = 0
    for batch_X, batch_Y in dataloader:
        batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_Y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == batch_Y).sum().item()
        
    acc = correct / len(dataset)
    if epoch % 5 == 0:
        print(f"Epoch [{epoch}/{EPOCHS}] - Loss: {total_loss/len(dataloader):.4f} - Acc: {acc:.4f}")

torch.save(model.state_dict(), SAVE_LSTM_PATH)
print(f"Đã lưu mô hình LSTM tại {SAVE_LSTM_PATH}")

Epoch [0/50] - Loss: 0.0970 - Acc: 0.9607
Epoch [5/50] - Loss: 0.0983 - Acc: 0.9609
Epoch [10/50] - Loss: 0.0859 - Acc: 0.9645
Epoch [15/50] - Loss: 0.0767 - Acc: 0.9686
Epoch [20/50] - Loss: 0.0769 - Acc: 0.9675
Epoch [25/50] - Loss: 0.0704 - Acc: 0.9706
Epoch [30/50] - Loss: 0.0746 - Acc: 0.9705
Epoch [35/50] - Loss: 0.0652 - Acc: 0.9743
Epoch [40/50] - Loss: 0.0619 - Acc: 0.9751
Epoch [45/50] - Loss: 0.0627 - Acc: 0.9743
Đã lưu mô hình LSTM tại /kaggle/working/lstm_intent.pth
